# Exp 4b — Cross-Graph Transfer + Spectral Analysis

<a href="https://colab.research.google.com/github/zixian0821-zoe/EN553744-final-project/blob/main/exp4b_transfer/04b_transfer.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Course:** EN.553.744 Data Science for Large-Scale Graphs (Spring 2026, JHU AMS)
**Team:** Yunwei Chai · Yang Song · Zixian Zhou

## What this notebook shows

Exp 4b probes **how transferable a ChebNet trained on one graph is when evaluated on another**, and explains the result through spectral geometry.

1. **3 x 3 transferability matrix.** Train on $\{S_{\text{src}}, S_{\text{tgt}}, S_{\text{fused}}\}$, evaluate on each of the same three. Diagonal = self-train; off-diagonal = transfer.
2. **User-degree stratification.** Re-run the same matrix on isolated-degree-0 users vs connected users.
3. **Feature ablation.** Replace source features with random noise to isolate the contribution of *topology* vs *features* to the transferred performance.
4. **Spectral distances.** Wasserstein-1 and L2 distances between the eigenvalue spectra of the three operators, plus operator Frobenius distances.

## Headline

$\text{NDCG@20}(\text{src} \to \text{tgt}) = 0.002292$, **93% of the target self-ceiling** ( $-6.9\%$ ). Transfer is nearly free, because the music and book operators are spectrally close: $W_1(S_{\text{src}}, S_{\text{tgt}}) = 0.023$.

Topology contributes $\approx 84\%$ of the transferred lift; features contribute $\approx 19\%$ — the *graph* carries most of the cross-domain signal.

## 0. Setup

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/EN553744_final_project/results/exp4b_transfer'
else:
    RESULTS_DIR = os.path.abspath('../../drive_upload/results/exp4b_transfer')

print('IN_COLAB    :', IN_COLAB)
print('RESULTS_DIR :', RESULTS_DIR)
print('exists      :', os.path.isdir(RESULTS_DIR))
if os.path.isdir(RESULTS_DIR):
    print('contents    :')
    for f in sorted(os.listdir(RESULTS_DIR)):
        print('  -', f)

## 1. Load result files

In [ ]:
def load_json(name):
    p = os.path.join(RESULTS_DIR, name)
    if not os.path.isfile(p):
        print(f'[missing] {name}')
        return None
    with open(p) as f:
        d = json.load(f)
    print(f'[loaded ] {name}')
    return d

transfer = load_json('transfer_reframed_results.json')   # 3x3 matrix + stratified + feature ablation
spectral = load_json('spectral_analysis_results.json')   # eigenvalue stats + Wasserstein distances

## 2. Aggregate 3 x 3 transferability matrix

Rows = train graph, columns = eval graph. Each cell is a fully-trained ChebNet (K=3) re-evaluated on a different ranking task. Higher is better.

In [ ]:
if transfer is None:
    raise SystemExit('transfer JSON not loaded')

GRAPHS = ['source', 'target', 'fused']
METRICS = ['NDCG@20', 'Recall@20', 'HitRate@20']

def matrix_for(metric):
    M = np.zeros((3, 3))
    for i, tr in enumerate(GRAPHS):
        for j, ev in enumerate(GRAPHS):
            M[i, j] = transfer['aggregate_matrix'][f'train={tr},eval={ev}'][metric]
    return M

for m in METRICS:
    M = matrix_for(m)
    df = pd.DataFrame(M, index=[f'train={g}' for g in GRAPHS],
                         columns=[f'eval={g}'  for g in GRAPHS])
    print(f'\n{m}  (model: {transfer["model"]})')
    print('-' * 50)
    print(df.round(6).to_string())

## 3. Heatmap of NDCG@20

In [ ]:
M = matrix_for('NDCG@20')

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(M, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(3)); ax.set_xticklabels([f'eval={g}' for g in GRAPHS])
ax.set_yticks(range(3)); ax.set_yticklabels([f'train={g}' for g in GRAPHS])
for i in range(3):
    for j in range(3):
        diag = ' (self)' if i == j else ''
        ax.text(j, i, f'{M[i,j]:.6f}{diag}', ha='center', va='center',
                color='black', fontsize=10)
fig.colorbar(im, ax=ax, label='NDCG@20')
ax.set_title(f'Exp 4b — Transferability matrix (NDCG@20)\nmodel: {transfer["model"]}')
plt.tight_layout()
plt.show()

## 4. Headline transfer numbers

In [ ]:
agg = transfer['aggregate_matrix']

src_to_tgt = agg['train=source,eval=target']['NDCG@20']
tgt_self   = agg['train=target,eval=target']['NDCG@20']
src_self   = agg['train=source,eval=source']['NDCG@20']
fused_self = agg['train=fused,eval=fused']['NDCG@20']

print('=' * 65)
print('TRANSFER HEADLINE  (NDCG@20)')
print('=' * 65)
print(f'  src -> src  (self-ceiling, source) : {src_self:.6f}')
print(f'  tgt -> tgt  (self-ceiling, target) : {tgt_self:.6f}')
print(f'  fused -> fused                     : {fused_self:.6f}')
print()
print(f'  src -> tgt  (cross-graph transfer) : {src_to_tgt:.6f}')
print(f'    fraction of target self-ceiling  : {src_to_tgt / tgt_self * 100:.1f}%')
print(f'    relative degradation             : {(src_to_tgt / tgt_self - 1) * 100:+.1f}%')
print()
print(f"  -> Source-trained ChebNet recovers {src_to_tgt / tgt_self * 100:.0f}% of the target self-ceiling.")
print(f'     Cross-domain transfer is nearly free.')

## 5. Stratified by user degree (isolated vs connected)

Same matrix, restricted to (a) users with no edges in the eval graph and (b) users with at least one edge. Tests whether the transfer effect is concentrated in well-connected users.

In [ ]:
for stratum in ['ALL', 'connected (deg>0)', 'isolated (deg=0)']:
    if stratum not in transfer['stratified_matrix']:
        continue
    cells = transfer['stratified_matrix'][stratum]
    M = np.zeros((3, 3))
    for i, tr in enumerate(GRAPHS):
        for j, ev in enumerate(GRAPHS):
            M[i, j] = cells[f'train={tr},eval={ev}']
    df = pd.DataFrame(M, index=[f'train={g}' for g in GRAPHS],
                         columns=[f'eval={g}'  for g in GRAPHS])
    print(f'\nNDCG@20 for stratum = "{stratum}"')
    print('-' * 55)
    print(df.round(6).to_string())

## 6. Feature ablation — topology vs features

What carries the transferred lift: the *shape* of the source graph, or the *content* of the source user features? Replace the source features with i.i.d. random noise and re-measure.

In [ ]:
fa = transfer.get('feature_ablation', {})

real_s2t   = fa.get('real_features_source_to_target')
random_s2t = fa.get('random_features_source_to_target')
random_s2s = fa.get('random_features_source_to_source')

if None in (real_s2t, random_s2t, random_s2s):
    print('feature_ablation block missing one or more keys')
else:
    print('=' * 65)
    print('FEATURE ABLATION  (NDCG@20, src -> tgt unless noted)')
    print('=' * 65)
    print(f'  real source features -> target : {real_s2t:.6f}')
    print(f'  RANDOM source features -> target: {random_s2t:.6f}')
    print(f'  RANDOM source features -> source: {random_s2s:.6f}   (sanity baseline)')
    print()
    feat_lift = real_s2t - random_s2t                     # what real features add
    topo_lift = random_s2t - random_s2s                   # what source topology adds vs random-baseline
    total     = max(real_s2t - random_s2s, 1e-9)
    print(f'  Decomposition of (real -> target) - (random src->src baseline):')
    print(f'    topology contribution         : {topo_lift:.6f}  ({topo_lift / total * 100:.1f}%)')
    print(f'    features contribution         : {feat_lift:.6f}  ({feat_lift / total * 100:.1f}%)')
    print(f'  -> Topology dominates feature content for cross-domain transfer.')

## 7. Operator (Frobenius) distances

How structurally different are the three normalized adjacency operators?

In [ ]:
od = transfer.get('operator_distances', {})

if not od:
    print('operator_distances missing')
else:
    print('OPERATOR FROBENIUS DISTANCES')
    print('-' * 40)
    for k, v in od.items():
        print(f'  {k:<22s} : {v:>10.4f}')
    print()
    print('  -> source and target are far apart in operator norm (40.8),')
    print('     but the fused operator sits exactly halfway (20.4 to each).')
    print('     The closeness of the SPECTRA explains transferability,')
    print('     not closeness of the operators themselves.')

## 8. Spectral analysis — eigenvalue stats and Wasserstein distances

In [ ]:
if spectral is None:
    print('[skip] spectral_analysis_results.json not loaded')
else:
    print('=' * 65)
    print(f'SPECTRAL ANALYSIS  (top {spectral["n_eigenvalues"]} eigenvalues, '
          f'{spectral["n_nodes"]} nodes)')
    print('=' * 65)
    print()
    print('Eigenvalue ranges per graph')
    print('-' * 55)
    print(f"{'graph':<10} {'min':>14} {'max':>14} {'mean':>14}")
    for g, r in spectral['eigenvalue_ranges'].items():
        print(f"{g:<10} {r['min']:>14.6f} {r['max']:>14.6f} {r['mean']:>14.6f}")

    print()
    print('Spectral distances between graphs')
    print('-' * 70)
    print(f"{'pair':<20} {'W_1':>10} {'L2 RMSE':>12} {'L2 total':>12} {'n_eig':>6}")
    for k, v in spectral['spectral_distances'].items():
        print(f"{k:<20} {v['wasserstein_1']:>10.4f} {v['l2_rmse']:>12.4f} "
              f"{v['l2_total']:>12.4f} {v['n_eigenvalues_compared']:>6d}")

    print()
    print('Operator Frobenius distances (spectral file copy)')
    print('-' * 50)
    for k, v in spectral['operator_frobenius_distances'].items():
        print(f'  {k:<22s} : {v:>10.4f}')

    print()
    w_st  = spectral['spectral_distances']['source-target']['wasserstein_1']
    w_sf  = spectral['spectral_distances']['source-fused']['wasserstein_1']
    w_tf  = spectral['spectral_distances']['target-fused']['wasserstein_1']
    print(f'  -> W_1(src, tgt) = {w_st:.4f}  (very small -> spectra are nearly identical)')
    print(f'     W_1(src, fused) = {w_sf:.4f}, W_1(tgt, fused) = {w_tf:.4f}')
    print(f'     Fused operator has a different spectrum from either parent (expected;')
    print(f'     it concentrates eigenvalues near 1 from the I + adjacency blend).')

## 9. Pre-rendered figures

In [ ]:
for fig_name in [
    'transfer_reframed_analysis.png',
    'distance_vs_degradation.png',
    'operator_distance_bars.png',
    'eigenvalue_distribution.png',
    'eigenvalue_zoom_50.png',
    'eigenvalue_cdf.png',
    'spectral_distance_wasserstein.png',
    'spectral_distance_l2.png',
    'spectral_energy.png',
]:
    p = os.path.join(RESULTS_DIR, fig_name)
    if os.path.isfile(p):
        print(f'>> {fig_name}')
        display(Image(p))
    else:
        print(f'[missing] {fig_name}')

## 10. Summary block (paste into report)

In [ ]:
if transfer and spectral:
    agg = transfer['aggregate_matrix']
    fa  = transfer.get('feature_ablation', {})
    sd  = spectral['spectral_distances']

    src_to_tgt = agg['train=source,eval=target']['NDCG@20']
    tgt_self   = agg['train=target,eval=target']['NDCG@20']
    real_s2t   = fa.get('real_features_source_to_target', float('nan'))
    rand_s2t   = fa.get('random_features_source_to_target', float('nan'))
    rand_s2s   = fa.get('random_features_source_to_source', float('nan'))
    total      = max(real_s2t - rand_s2s, 1e-9)

    print('=' * 65)
    print('SUMMARY FOR REPORT  (Exp 4b — Transfer + Spectral)')
    print('=' * 65)
    print(f"""
Setup
  Model         : {transfer['model']}
  Train graphs  : source, target, fused
  Eval graphs   : source, target, fused (3 x 3 transferability matrix)
  Stratification: ALL, connected (deg>0), isolated (deg=0)
  Ablations     : real vs random source features
  Spectral      : top {spectral['n_eigenvalues']} eigenvalues, W_1 + L2 distances

Transferability headline (NDCG@20)
  src -> tgt          : {src_to_tgt:.6f}
  tgt -> tgt (ceiling): {tgt_self:.6f}
  fraction of ceiling : {src_to_tgt / tgt_self * 100:.1f}%   (degradation only {(1 - src_to_tgt/tgt_self) * 100:.1f}%)

Topology vs features decomposition
  real src features  -> tgt    : {real_s2t:.6f}
  random src features -> tgt   : {rand_s2t:.6f}
  random src features -> src   : {rand_s2s:.6f}    (baseline)
  Topology contribution        : {(rand_s2t - rand_s2s) / total * 100:.1f}%
  Features contribution        : {(real_s2t - rand_s2t) / total * 100:.1f}%

Spectral distances (Wasserstein-1, top {spectral['n_eigenvalues']} eigenvalues)
  W_1(src,    tgt)  : {sd['source-target']['wasserstein_1']:.4f}
  W_1(src,    fused): {sd['source-fused']['wasserstein_1']:.4f}
  W_1(tgt,    fused): {sd['target-fused']['wasserstein_1']:.4f}

Conclusion
  Source-trained models transfer to target with negligible loss
  ({(1 - src_to_tgt/tgt_self) * 100:.1f}%), because the source and target Laplacian spectra
  are nearly identical (W_1 = {sd['source-target']['wasserstein_1']:.3f}, despite an operator-norm
  distance of {transfer['operator_distances']['source-target']:.1f}). The decomposition shows topology
  carries the bulk of the cross-domain signal, with features providing
  a smaller refinement. This is the empirical justification for the
  fused-graph approach in Exp 2: cross-domain prediction is feasible
  because cross-domain spectra align.
""")